# Chapter 23 — Playground & Prompts Hub

Hands-On LangChain. Uses the **LangSmith** SDK.

As an agent evolves, hardcoding prompts directly in your code becomes a problem: what happens when you
want to experiment with variations, roll back a change, or let a non-engineer tweak the wording? The
LangSmith **Prompt Hub** and **Playground** solve this — they separate prompt engineering from code
deployment. The **Hub** is a centralized, **versioned** repository for prompts; the **Playground** is an
interactive place to test and compare them. Once a prompt is good, you push it to the Hub and **pull it
into your code** by name — no redeploy needed to change a prompt.

Companion notebook for Chapter 23.

> **Requires a LangSmith account** (`LANGSMITH_API_KEY` in `.env`) and a running Ollama server with `qwen2.5:7b` available (or another model selected with `MODEL` in `.env`). The runnable section pushes a prompt to *your* Hub and pulls it back.
> ```bash
> ollama pull qwen2.5:7b
> pip install langsmith langchain-ollama langchain-community ddgs python-dotenv -q
> ```

## 23.1 The Prompt Hub & Playground

When you open the **Playground** (in the LangSmith UI), you can edit a prompt and immediately see how the
output changes, test different LLMs from different providers side by side, adjust parameters in real time,
and watch token usage and cost update instantly. A common cleanup step — which we'll rely on below — is to
pull the variable parts (the user query, the search results) *out* of the prompt and make them **inputs**,
leaving a neat reusable system prompt. When you're happy with it, you **save (push)** it to the Hub.

*(Figures 23.1–23.3 are screenshots of the LangSmith web UI.)*

## 23.2 Prompt versioning

One of the Hub's most powerful features is **automatic versioning**. Every time you push a prompt,
LangSmith generates a unique **commit hash** capturing that exact version — the template, its variables,
and any model configuration. You can experiment freely knowing you can roll back; and because traces
record which prompt version produced each run, you can correlate output quality with specific prompt
changes (and even A/B test versions).

Let's make this concrete by **pushing a prompt to your Hub from code**. We define a `ChatPromptTemplate`
with two variables — `question` and `context` — and push it under the name `recipe_discovery_prompt`.

In [1]:
import os, warnings
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
warnings.filterwarnings("ignore")

# tracing config FIRST
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "recipe-discovery-agent"

from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

recipe_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a professional chef and nutritionist. Your job is to help people discover and understand "
     "recipes based on their interests.\n\nBased on the search results below, provide:\n"
     "1. A quick overview of the dish/cuisine\n2. Key ingredients typically used\n"
     "3. Difficulty level (beginner/intermediate/advanced)\n4. Cooking time estimate\n"
     "5. One helpful cooking tip\n\nQuery: {question}\n\nSearch Results: {context}"),
    ("human", "{question}"),
])

client = Client()
commit_url = client.push_prompt("recipe_discovery_prompt", object=recipe_prompt)
print("Pushed. Commit URL:")
print(commit_url)

Pushed. Commit URL:
https://smith.langchain.com/prompts/recipe_discovery_prompt/a6980220?organizationId=e6f89334-7467-5f93-8a8c-8adbd440f03a


## 23.3 Pulling prompts from LangSmith

Now we rebuild the recipe agent from Chapter 22 — but instead of **hardcoding** the prompt, we **pull it
from the Hub** by name. We initialize a client and call `pull_prompt`.

In [2]:
pulled_prompt = client.pull_prompt("recipe_discovery_prompt")

print("Type           :", type(pulled_prompt).__name__)
print("Input variables:", pulled_prompt.input_variables)
print("Commit hash     :", pulled_prompt.metadata.get("lc_hub_commit_hash", "")[:16], "...")

Type           : ChatPromptTemplate
Input variables: ['context', 'question']
Commit hash     : a69802209cc8af42 ...


The pulled object isn't just a string — it's a complete **`ChatPromptTemplate`** with structured
system/user messages, and **metadata** recording exactly which version (commit hash) you're using. That
traceability means if a prompt version causes problems in production, you can identify and roll back
instantly. Note the variables are `question` and `context` — standardized names from the centralized
prompt, rather than whatever each agent happened to call them.

Invoking it is simple: pipe the pulled prompt into a model and call `invoke` with a dict of variables —
the template handles all the message formatting.

In [ ]:
from model_provider import get_chat_model
from langchain_community.tools import DuckDuckGoSearchResults
from langsmith import traceable

llm = get_chat_model(temperature=1)
search_tool = DuckDuckGoSearchResults(num_results=3, output_format="list")

@traceable
def search_recipes(query):
    """Search for recipe information."""
    try:
        results = search_tool.invoke({"query": f"{query} recipe"})
        return "\n".join(str(i) for i in results) if isinstance(results, list) else str(results)
    except Exception as e:
        return f"Could not find recipe information: {e}"

@traceable
def create_recipe_guide(query, context):
    """Generate a guide using the prompt pulled from the LangSmith Hub."""
    chain = pulled_prompt | llm          # the Hub prompt drives the model
    return chain.invoke({"question": query, "context": context}).content

@traceable
def recipe_assistant(query):
    """Main function: search recipes and create a guide."""
    context = search_recipes(query)
    return create_recipe_guide(query, context)

Run the agent. It behaves like Chapter 22's, but its prompt now lives in the Hub — we could change the prompt there without touching this code.

In [ ]:
result = recipe_assistant("Thai green curry")
print(result[:900])

We got a recipe guide using the **same prompt as before, this time pulled from the Hub** — and the
whole run was traced to your `recipe-discovery-agent` project. The payoff: a teammate (or you) can refine
this prompt in the Playground and push a new version, and every deployment that pulls
`recipe_discovery_prompt` picks it up — **no code change, no redeploy**.

## 23.4 Prompt Canvas: auto-optimizing your prompts

A final Hub feature worth knowing is **Prompt Canvas**. Instead of hand-editing, you give an LLM agent
**instructions** ("make it shorter," "target a beginner reading level," or a custom action) and it
rewrites the prompt for you — either the whole thing or just a selected part — with a diff view so you can
see exactly what changed before committing. The goal, in LangSmith's words, is to make everyone on the
team "a professional prompter."

Together, the Playground, Hub, and Prompt Canvas turn prompt development from scattered trial-and-error
into a **systematic, observable, collaborative** process — with full version history and the confidence
to iterate rapidly.

---

This completes our hands-on tour of LangSmith's three pillars across this part: **observability** (the
overview in Chapter 21), **tracing** (Chapter 22), and **prompt engineering** (this chapter). Together
they give you the tools to build, debug, and continuously improve LLM applications with confidence.